# testLeague — loading a season through the ESPN adapter

Loads a season with `EspnLeagueRepository` and looks at the resulting domain objects
(`League`, `Team`, `Matchup`, `Lineup`), plus a few sanity checks against ESPN's own numbers.

Tables come from `fantasy_analysis.application.frames`: `teams_frame`, `matchups_frame`,
`team_results_frame`, `standings_frame`, `schedule_frame`, `lineup_frame`,
`lineup_entries_frame`, `bench_points_frame`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from collections import Counter

import pandas as pd

from fantasy_analysis.application.frames import (
    bench_points_frame,
    lineup_entries_frame,
    lineup_frame,
    matchups_frame,
    schedule_frame,
    standings_frame,
    team_results_frame,
    teams_frame,
)
from fantasy_analysis.config import settings
from fantasy_analysis.infrastructure.espn import EspnLeagueRepository

pd.set_option("display.max_columns", 50)

SEASON = 2026

## Load the season

Makes several ESPN requests per week, so a full season takes a little while.

In [ ]:
repo = EspnLeagueRepository.from_settings(settings)
league = repo.get_league(SEASON)

weeks = sorted({m.week for m in league.matchups})
print(f"{league.name} ({league.season}): {league.num_teams} teams, weeks {weeks[0]}-{weeks[-1]}")
print("matchups by kind:", dict(Counter(m.kind for m in league.matchups)))

In [ ]:
league

## Teams

In [ ]:
teams_frame(league)

## Matchups

In [ ]:
matchups_frame(league)

## Standings

Regular season by default; pass `kind="PLAYOFF"` or `"CONSOLATION"` for the others.

In [ ]:
standings_frame(league)

## Team results

One row per team per game: both sides of every matchup.

In [ ]:
team_results_frame(league)

## One team's week: lineup with actual vs. projected points

In [ ]:
weeks

In [ ]:
team = league.teams[9]
WEEK = 7
lineup = team.lineups[WEEK]

print(f"{team.name}, week {WEEK}: {lineup.points:.2f} points from starters")
lineup_frame(lineup)

In [ ]:
# That team's schedule
schedule_frame(league, team)

## Every lineup entry, season-long

One row per player per team per week — the main table for analysis.

In [ ]:
entries_df = lineup_entries_frame(league)
print(entries_df.shape)
entries_df.head()

In [ ]:
# Points left on the bench, per team
bench_points_frame(league)

## Sanity checks

In [ ]:
# ESPN's matchup score should equal the sum of starters' points
mismatches = [
    (m.week, side.name, score, round(side.lineups[m.week].points, 2))
    for m in league.matchups
    for side, score in ((m.home, m.home_score), (m.away, m.away_score))
    if side is not None and score is not None and abs(score - side.lineups[m.week].points) > 0.01
]
print("score mismatches:", mismatches or "none")

# Every team should have a lineup in every week it played
missing = [
    (m.week, side.name)
    for m in league.matchups
    for side in (m.home, m.away)
    if side is not None and m.week not in side.lineups
]
print("missing lineups:", missing or "none")

# Lineups with an empty starting slot (bench > 6)
short = entries_df[entries_df["starter"]].groupby(["week", "team"]).size()
print("lineups with an empty starting slot:")
short[short < 10]